# Correlation, Causation & Confounding

Companion notebook for the [Correlation & Confounding lesson](https://ml-viz-ruby.vercel.app/courses/causal-inference/01-correlation-and-confounding).

**The idea in one sentence.** A **confounder** — a common cause of both the
treatment and the outcome — can manufacture a correlation where there is no
causal effect at all, which is why "correlation ≠ causation" and why you must
**adjust** for confounders before believing a number.

The classic trap simulated here: healthier patients are both *more likely to get
treated* and *more likely to recover on their own*, so a naive
treated-minus-untreated comparison credits the treatment for the health it never
caused. Stratifying by health reveals the true (zero) effect — and **Simpson's
paradox** shows the pooled association can even *reverse* the within-group truth.

We build it from scratch, **validate that adjustment recovers the true effect
(and cross-check with a regression)**, then cover the gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## 1 — A confounder fakes an effect

Health Z drives BOTH who gets treated and who recovers. The treatment itself does nothing (true
effect = 0), yet the naive treated-vs-untreated comparison shows a big 'effect'.

In [ ]:
n = 5000
Z = rng.random(n)                                  # latent health in [0,1]
T = (rng.random(n) < Z).astype(int)                # healthier patients more likely treated
# recovery depends ONLY on health Z, not on treatment T (true effect = 0)
Y = (rng.random(n) < Z).astype(int)

naive = Y[T==1].mean() - Y[T==0].mean()
print(f'true causal effect of treatment: 0.000 (by construction)')
print(f'naive treated - untreated:       {naive:+.3f}  <- looks like the treatment helps a lot!')

## 2 — Adjusting for the confounder reveals the truth

Compare treated vs untreated *within* strata of Z, then average over Z (the backdoor adjustment).
The fake effect collapses to ~0.

In [ ]:
def adjusted_effect(Z, T, Y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    eff, wts = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (Z >= lo) & (Z < hi)
        if (T[m]==1).sum() and (T[m]==0).sum():
            eff.append(Y[m & (T==1)].mean() - Y[m & (T==0)].mean())
            wts.append(m.sum())
    return np.average(eff, weights=wts)

print(f'naive effect:               {naive:+.3f}')
print(f'confounder-adjusted effect: {adjusted_effect(Z, T, Y):+.3f}  <- ~0, the truth')

### Validate: adjustment recovers the true (zero) effect

The treatment's true effect is **0** by construction (recovery depends only on
health $Z$). We assert the naive estimate is badly biased upward while the
confounder-adjusted estimate is near zero — and cross-check with a **regression**
of $Y$ on both $T$ and $Z$, whose coefficient on $T$ should also be ~0.

In [ ]:
from sklearn.linear_model import LinearRegression

adj = adjusted_effect(Z, T, Y)
print(f'naive estimate:            {naive:+.3f}  (should be far from 0 — biased)')
print(f'stratification-adjusted:   {adj:+.3f}  (should be ~0 — the truth)')
assert abs(naive) > 0.1, 'the confounder should create a large spurious effect'
assert abs(adj) < 0.05, 'adjusting for Z should recover the true zero effect'

# regression adjustment: include Z as a covariate; the coefficient on T is the effect
X = np.column_stack([T, Z])
coef_T = LinearRegression().fit(X, Y).coef_[0]
print(f'\nregression coefficient on T (adjusting for Z): {coef_T:+.3f}  (~0)')
assert abs(coef_T) < 0.05, 'regression that controls for Z also finds ~0 effect'
print('\n✅ both stratification and regression adjustment recover the true zero effect')

## 3 — Simpson's paradox: the association reverses

We build data where, overall, the treated recover LESS, but within every subgroup they recover MORE
— because the treatment was given more often in the harder subgroup.

In [ ]:
# subgroup A (easy, high base recovery), subgroup B (hard, low base recovery)
def make_group(n, base, treat_rate, boost):
    T = (rng.random(n) < treat_rate).astype(int)
    Y = (rng.random(n) < base + boost * T).astype(int)   # treatment genuinely helps (+boost)
    return T, Y

Ta, Ya = make_group(3000, base=0.80, treat_rate=0.15, boost=0.05)   # easy, rarely treated
Tb, Yb = make_group(3000, base=0.20, treat_rate=0.85, boost=0.05)   # hard, usually treated
T = np.r_[Ta, Tb]; Y = np.r_[Ya, Yb]

print('WITHIN subgroup A: treated - untreated =', round(Ya[Ta==1].mean() - Ya[Ta==0].mean(), 3))
print('WITHIN subgroup B: treated - untreated =', round(Yb[Tb==1].mean() - Yb[Tb==0].mean(), 3))
print('OVERALL (pooled):  treated - untreated =', round(Y[T==1].mean() - Y[T==0].mean(), 3),
      ' <- reversed sign!')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **unobserved confounders** | you can only adjust for what you measure; hidden common causes still bias you |
| **collider bias** | conditioning on a common *effect* induces a spurious association (demo below) |
| **Simpson's paradox** | pooled and subgroup effects can have opposite signs |
| **overlap / positivity** | a stratum with only-treated or only-untreated units can't estimate a within-stratum effect |
| **"control for everything" is wrong** | the correct adjustment set comes from the causal DAG, not from throwing in all variables |

Demo: adjusting for a **collider** *creates* bias rather than removing it.

In [ ]:
# The danger of adjusting for the WRONG variable: conditioning on a COLLIDER (a common
# EFFECT of T and Y) creates bias instead of removing it. Here C is caused by both T and Y;
# controlling for it induces a spurious T-Y association where none existed.
n2 = 20000
T2 = (rng.random(n2) < 0.5).astype(int)
Y2 = (rng.random(n2) < 0.5).astype(int)        # T2 and Y2 are INDEPENDENT (true effect 0)
C  = ((T2 + Y2 + (rng.random(n2) < 0.1)) >= 1).astype(int)   # collider: common effect
# unconditional association ~ 0; conditional-on-collider association != 0
uncond = Y2[T2==1].mean() - Y2[T2==0].mean()
within_C1 = Y2[(T2==1)&(C==1)].mean() - Y2[(T2==0)&(C==1)].mean()
print(f'T–Y effect, unconditional:        {uncond:+.3f}  (~0, correct)')
print(f'T–Y effect, conditioned on C=1:   {within_C1:+.3f}  (spurious! collider bias)')
print('\nAdjusting is not always good: conditioning on a collider CREATES bias. Use a causal graph.')

## ✏️ Your turn

**Exercise.** Implement `naive_effect(T, Y)` (treated mean minus untreated mean) and
`stratified_effect(group, T, Y)` — the treatment effect averaged within each subgroup, weighted by
subgroup size. The gap between them is the confounding bias.

In [ ]:
def naive_effect(T, Y):
    # TODO(you): mean outcome of treated minus mean outcome of untreated
    return ...

def stratified_effect(group, T, Y):
    # TODO(you): within-subgroup (treated-untreated) effect, averaged weighted by subgroup size
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
group = np.r_[np.zeros(3000), np.ones(3000)]      # subgroup label A=0, B=1
assert np.isclose(naive_effect(T, Y), Y[T==1].mean() - Y[T==0].mean())
# pooled effect is negative (Simpson) but the stratified effect is positive (the truth)
assert naive_effect(T, Y) < 0
assert stratified_effect(group, T, Y) > 0
print(f'\u2713 naive={naive_effect(T,Y):+.3f} (misleading)  stratified={stratified_effect(group,T,Y):+.3f} (correct)')

<details>
<summary>Solution</summary>

```python
def naive_effect(T, Y):
    return Y[T==1].mean() - Y[T==0].mean()

def stratified_effect(group, T, Y):
    effs, wts = [], []
    for g in np.unique(group):
        m = group == g
        effs.append(Y[m & (T==1)].mean() - Y[m & (T==0)].mean())
        wts.append(m.sum())
    return np.average(effs, weights=wts)
```

The pooled and stratified estimates disagree because the subgroup is a confounder (it affects both
treatment assignment and outcome). Which estimate is *correct* depends on the causal structure —
here the subgroup is a confounder, so the stratified (adjusted) effect is the right one.

</details>

## Key takeaways

- **Confounders fake effects.** A common cause of treatment and outcome makes a
  naive comparison credit the treatment for something it didn't do.
- **Adjust for confounders** (stratify or regress with $Z$ as a covariate) to
  recover the truth — we verified both give ~0 here.
- **Simpson's paradox:** the pooled association can reverse the sign of every
  subgroup's — always ask "adjusted for what?"
- **But don't adjust blindly.** Conditioning on a **collider** (a common *effect*)
  *creates* bias. Which variables to adjust for is a question about the causal
  graph, not the data.